# TiRex-2 Multivariate Coupling

The TiRex-2 coupling pipeline (Podest et al., 2026) builds a **multivariate time
series** by first generating independent univariate series, then applying a
randomly chosen coupling mechanism, and finally post-processing:

1. independent univariate generation (+ stage-1 augmentation);
2. coupling via one of 7 mechanisms;
3. post-processing (time warping, masking, discretization, ...).

The 7 mechanisms are: ``identity``, ``univariate``, ``functional``,
``linear_mixing``, ``cointegration``, ``linear_scm``, ``nonlinear_scm``.

Reference: Podest, P., et al. (2026). *TiRex-2*. arXiv:2607.01204, Section 3.4.

In [ ]:
import os
import sys

import numpy as np

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from s2generator.scm import CouplingPipeline

In [ ]:
rng = np.random.RandomState(0)
pipe = CouplingPipeline()

## Generate a coupled series from scratch

``generate`` builds the base series, augments them, couples them and post-processes.
The result is a series of shape $(T, Q)$.

In [ ]:
x = pipe.generate(rng, n_inputs_points=128, input_dimension=4)
print("coupled series shape:", x.shape)

## Available mechanisms

In [ ]:
print("mechanisms:", list(pipe.mechanisms.keys()))

## Fix a specific mechanism

In [ ]:
x = pipe.generate(rng, 128, input_dimension=4, mechanism="cointegration")
print("cointegration shape:", x.shape)

## Couple an existing input series (``__call__``)

In [ ]:
series = rng.normal(0, 1, (128, 4))
coupled = pipe(rng, series, mechanism="linear_mixing", apply_postprocessing=False)
print("input:", series.shape, " -> coupled:", coupled.shape)

## Classification label

Passing ``n_classes`` returns $(x, y)$; ``generate_batch`` yields balanced labels.

In [ ]:
x, y = pipe.generate(rng, 128, input_dimension=4, n_classes=3)
print("series shape:", x.shape, " label:", y)

batch = pipe.generate_batch(
    rng, n_samples=30, n_inputs_points=128, input_dimension=4, n_classes=3
)
labels = [lab for _, lab in batch]
print("class counts:", np.bincount(labels))

## Custom DAG for the SCM mechanisms

``adjacency`` fixes the parent structure over the $Q$ variates; it only affects the
``linear_scm`` / ``nonlinear_scm`` mechanisms (the others ignore it). When
``input_dimension`` is omitted, the variate count is inferred from the graph.

In [ ]:
Q = 4
adj = np.zeros((Q, Q), dtype=bool)
for i in range(Q - 1):
    adj[i, i + 1] = True

x = pipe.generate(rng, 128, mechanism="linear_scm", adjacency=adj)
print("shape with custom graph:", x.shape)

## Visualize generated sequences

One line plot per coupled sequence: each variate is drawn as a line over time.
The cell below generates a small batch and shows each in its own subplot.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Generate a few coupled series (4 variates each) to inspect. Post-processing is
# disabled here so every series keeps a clean, uniform (T, Q) shape.
rng = np.random.RandomState(0)
seqs = [
    pipe.generate(rng, n_inputs_points=128, input_dimension=4,
                  apply_postprocessing=False)
    for _ in range(4)
]

n = len(seqs)
ncols = 2
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3 * nrows), squeeze=False)
for i, s in enumerate(seqs):
    ax = axes[i // ncols][i % ncols]
    ax.plot(s)  # (T, Q): one line per variate
    ax.set_title(f"series {i}  ({s.shape[1]} variates)")
    ax.set_xlabel("time step")
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
fig.tight_layout()
plt.show()